This notebook can be used to generate the main graphs in our paper on mechanotransduction signaling in cells on nanopillars.

First, we introduce the necessary imports and configure our matplotlib settings.

In [ ]:
import sys, os, pathlib
sys.path.append("/root/shared/gitrepos/smart-mechanotransduction/utils")
sys.path.append("/root/shared/gitrepos/smart-comp-sci/mechanotransduction-example")
cur_dir = str(pathlib.Path.cwd() / "..")
import smart_analysis
from matplotlib import pyplot as plt
import matplotlib
plt.style.use(f"smart_plots.mplstyle")
matplotlib.rcParams.update({'figure.figsize': (5,4)})
import numpy as np

This runs data analysis on all simulations in all subfolders within the folder specified by `files_dir` and saves into `npy_dir`. The correct mesh must be included in a subfolder of `mesh_dir`.

In [ ]:
import re
mesh_dir = "/root/shared/gitrepos/smart-nanopillars/meshes/nanopillars_baseline"
files_dir = "/root/scratch/new_coupled_sims/results_nanopillars_pt5Lamin_2026-01-30_newPhos_phiE"
npy_dir = pathlib.Path.cwd() / ".." / "analysis_data" / "npy-files-pt5Lamin-submission"
npy_dir.mkdir(exist_ok=True)
test_folders = os.listdir(files_dir)
condition_str = []
for i in range(len(test_folders)):
    mesh_file = f"{files_dir}/{test_folders[i]}/mesh/nuc_mesh.h5"
    # for folder in os.listdir(mesh_dir):
    #     if folder[12:] in test_folders[i]: # mesh name matches current case
    #         mesh_file = f"{mesh_dir}/{folder}/spreadCell_mesh.h5"
    #         break
    # if mesh_file == "":
    #     print("Mesh could not be found, skipping to next case")
    #     condition_str.append("")
    #     continue
    if not (("force400.0" in test_folders[i] or "force0.0" in test_folders[i]) and 
            "diff0.001" in test_folders[i]):
        continue
    results_folder = f"{files_dir}/{test_folders[i]}"
    condition_cur = test_folders[i]
    condition_str.append(condition_cur)
    if mesh_file=="" or results_folder=="":
        raise ValueError("Folders do not match expected structure for analysis")
    height = 1.04
    zmax_nuc = height + 4.1*2
    tests = ["center_low"]#,"center_high"] # other spatial domains could be specified here
    center_low = [-1, -1, 0, 1, 1, height+1]
    center_high = [-1, -1, zmax_nuc-1.0, 1, 1, zmax_nuc+10]
    domains = [center_low, center_high]
    for sd in range(len(tests)):
        # try:
        tVec, results_all = smart_analysis.analyze_all(
                mesh_file=mesh_file, results_path=results_folder, display=False, axisymm=False,
                subdomain=domains[sd], jacobians=True)
        #     print("error in analysis, skipping to next case")
        # except:
        #     continue
        results_all.insert(0, tVec) # add time as first element in list
        max_length = len(tVec)
        for j in range(len(results_all)):
            if len(results_all[j]) > max_length:
                max_length = len(results_all[j])
        for j in range(len(results_all)):
            num_zeros = max_length - len(results_all[j])
            for k in range(num_zeros):
                results_all[j].append(0)
        np.save(npy_dir / f"{condition_cur}_results_{tests[sd]}.npy", results_all)

Generate graphs of YAP/TAZ N/C over time for cells on different nanopillar substrates (Fig 2D) and store relevant steady-state concentrations.

In [ ]:
from smart import mesh_tools
import dolfin as d

# laminDiff = 0.001 # or .01
# pitchArray=[0.0, 1.5, 3.0, 1.5, 3.0]
# radiusArray=[0.0, 0.2, 0.2, 0.5, 0.5]
# forceArray=[0.0, 400.0, 400.0, 800.0, 800.0, 0.0, 400.0, 400.0, 800.0, 800.0]
# t0Array=[100.0, 100.0, 1000.0, 100.0, 1000.0, 100.0, 100.0, 1000.0, 100.0, 1000.0]
# actinArray=[0.0, 0.0, 0.0, 0.0, 0.0, 20.0, 20.0, 20.0, 20.0, 20.0]
pitchArray = [0.0, 3.0, 3.0]
radiusArray = [0.0, 0.2, 0.5]
substrate_names = ["flat", "200 nm NP", "500 nm NP"]
forceArray = [0.0, 100.0, 200.0, 300.0, 400.0, 500.0, 600.0, 700.0, 800.0]
var_names_saved = ["aNPC", "FActin", "J_NE", "J_Nuc", "LaminA", "LaminAp", "NE_alt_udef", "NE_udef", 
                   "NPC", "NPC_A", "Nuc_udef", "rupture_marker", "YAPTAZ_nuc"]
plot_names = ["YAPTAZ_nuc", "LaminA", "LaminAp", "rupture_marker", "NPC_A"]
results_idx_ref = []
for name in plot_names:
    results_idx_ref.append(var_names_saved.index(name) + 1) # add one because time is first el

YAPTAZ_ratios = np.zeros([len(pitchArray),len(forceArray)])
vol_vals = np.zeros([len(pitchArray),len(forceArray)])
sa_vals = np.zeros([len(pitchArray),len(forceArray)])
vol_data = [[],[],[]]

plotDyn = False
if plotDyn:
    matplotlib.rcParams.update({'figure.figsize': (6,8)})
else:
    # matplotlib.rcParams.update({'figure.figsize': (3.5,4)})
    matplotlib.rcParams.update({'figure.figsize': (5,4)})

for phiE in [0,2]:
    for i in range(len(pitchArray)):
        for j in range(len(forceArray)):
            if phiE==1:
                results_idx = results_idx_ref
                cur_folder = (f"/root/scratch/new_coupled_sims/results_nanopillars_sweepForce_2026-01-31_scalept2_phiE/"
                            f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                            f"force{forceArray[j]}")#_t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}")
                file_cur = (f"{cur_dir}/analysis_data/npy-files-sweepForce-phiE-submission/"#_newPhos_scalept2_phiE_newAdvectAndDiffuse/"#nuc-mechanics-coupled-corrected/"
                    f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                    f"force{forceArray[j]}_results_all.npy")#t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}_results_all.npy")
            elif phiE==0:
                results_idx = results_idx_ref
                cur_folder = (f"/root/scratch/new_coupled_sims/results_nanopillars_sweepForce_2026-01-31_scalept2/"
                            f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                            f"force{forceArray[j]}")#_t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}")
                file_cur = (f"{cur_dir}/analysis_data/npy-files-sweepForce-submission/"#nuc-mechanics-coupled-corrected/"
                    f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                    f"force{forceArray[j]}_results_all.npy")#t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}_results_all.npy")
            elif phiE==2:
                results_idx = np.array(results_idx_ref)-1
                cur_folder = (f"/root/scratch/new_coupled_sims/results_nanopillars_sweepForce_2026-01-31_scalept2_noStretchSens/"
                            f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                            f"force{forceArray[j]}")#_t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}")
                file_cur = (f"{cur_dir}/analysis_data/npy-files-sweepForce-noStretchSens-submission/"#nuc-mechanics-coupled-corrected/"
                    f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                    f"force{forceArray[j]}_results_all.npy")#t0{t0Array[j]}_diff{laminDiff}_actin{actinArray[j]}_results_all.npy")
            # loaded_cur = mesh_tools.load_mesh(pathlib.Path(f"{cur_folder}/mesh/nuc_mesh.h5"))
            YAPTAZ_cur = np.loadtxt(pathlib.Path(f"{cur_folder}/YAPTAZ.txt"))
            vol_inner = np.loadtxt(pathlib.Path(f"{cur_folder}/mech_data/vol_inner.txt"))
            vol = np.loadtxt(pathlib.Path(f"{cur_folder}/mech_data/vol.txt"))
            sa_inner = np.loadtxt(pathlib.Path(f"{cur_folder}/mech_data/inner_SA.txt"))
            sa_outer = np.loadtxt(pathlib.Path(f"{cur_folder}/mech_data/outer_SA.txt"))
            
            results_cur = np.load(file_cur)
            YAPnuc = results_cur[results_idx[0]]
            if radiusArray[i] == 0.0:
                FActin0 = 140
            elif radiusArray[i] <= 0.15:
                FActin0 = 80.0
            elif radiusArray[i] <= 0.25:
                FActin0 = 100.0
            elif radiusArray[i] <= 0.5:
                FActin0 = 120.0
            else:
                FActin0 = 140.0
            Myo_A = 4.0
            K_CN = 4.0
            K_CY = 5.429e-3
            phiNP = (K_CN + K_CY*FActin0*Myo_A)/(1+ K_CN + K_CY*FActin0*Myo_A)
            YAPcyto = YAPTAZ_cur[:,1]/phiNP
            maxIdx = min([len(YAPnuc),len(YAPcyto)])
            YAPratio = YAPnuc[0:maxIdx] / YAPcyto[0:maxIdx]
            LaminA = results_cur[results_idx[1]]
            YAPTAZ_ratios[i,j] = YAPratio[-1]
            vol_vals[i,j] = vol_inner[-1]
            sa_vals[i,j] = sa_inner[-1]
            # plt.plot(results_cur[0], YAPratio,
            #         label=f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_force{forceArray[j]}")#_t0{t0Array[j]}_actin{actinArray[j]}")
            start_vol = len(vol) - len(YAPratio)
            vol_tot = (vol[start_vol:]+vol_inner[start_vol+1:])
            if j == 0:
                YAPnuc_ref = YAPnuc[-1]
            # print(YAPratio[0])
            # print(vol_tot[0])
            if plotDyn:
                plt.subplot(411)
                plt.semilogx(results_cur[0], vol_tot)#YAPratio[0]*(vol_tot[0]/vol_tot),'--')
                plt.ylabel('Nuclear volume')
                plt.xlim([1, 10000])
                plt.subplot(412)
                plt.semilogx(results_cur[0], sa_outer[start_vol:])
                plt.ylabel('NE area')
                plt.xlim([1, 10000])
                plt.subplot(413)
                plt.semilogx(results_cur[0], results_cur[results_idx[4]])
                plt.ylabel('Active NPC')
                plt.xlim([1, 10000])
                plt.subplot(414)
                plt.semilogx(results_cur[0], results_cur[results_idx[0]])
                plt.ylabel('YAP/TAZ')
                plt.xlim([1, 10000])
            print(f"{i},{j}")
        # plt.plot(vol_vals[i,:], sa_vals[i,:], 'o', label=substrate_names[i])
        if not plotDyn:
            if phiE == 0:
                plt.plot(vol_vals[i,:]/150.7, YAPTAZ_ratios[i,:], 'o', fillstyle='none', label=substrate_names[i])
            elif phiE == 1:
                plt.plot(vol_vals[i,:]/150.7, YAPTAZ_ratios[i,:], 'o', label=substrate_names[i])
            elif phiE == 2:
                plt.plot(vol_vals[i,:]/150.7, YAPTAZ_ratios[i,:], 's', fillstyle='none', label=substrate_names[i])
        vol_NE = np.average(vol)
        vol_cell = 1024#vol_vals[i,0] + 437*2 #1024
        Nconv = 602.2
        NY_tot = 540000#Nconv*(0.9*437*2 + 0.7*150)#(vol_vals[i,0]))#533900
        # NY_nuc = YAPnuc_ref*(vol_vals[i,0])*Nconv # initial value to number
        # YAPpred = ((vol_cell - vol_vals[i,:])/(vol_vals[i,:])) * (NY_nuc / (NY_tot - NY_nuc))
        # plt.plot(vol_vals[i,:]/154.4, YAPpred, label=substrate_names[i])
        # plt.plot(forceArray, vol_vals[i,:], 'o', label=substrate_names[i])
    vol_data[phiE] = vol_vals.copy()
# plt.legend()
# plt.ylabel("YAP/TAZ N/C")
# plt.xlabel('Time (s)')
koushki_data = np.loadtxt("koushki_data.txt")
if not plotDyn:
    # plt.plot(koushki_data[:,0]/2136.114733, koushki_data[:,2], 'x', label="Koushki et al 2023")
    plt.xlim([0.45, 1.05])
    plt.ylim([0.55, 1.5])
    plt.xlabel('Relative volume')
    plt.ylabel('YAP/TAZ N/C')
plt.legend()
plt.savefig("YAPTAZCorr_noStretchSens.pdf", format="pdf")

In [ ]:
plt.plot(forceArray, vol_data[1][0,:]-vol_data[0][0,:])
plt.plot(forceArray, vol_data[2][0,:]-vol_data[0][0,:])
plt.plot(forceArray, vol_data[1][1,:]-vol_data[0][1,:])
plt.plot(forceArray, vol_data[2][1,:]-vol_data[0][1,:])
plt.plot(forceArray, vol_data[1][2,:]-vol_data[0][2,:])
plt.plot(forceArray, vol_data[2][2,:]-vol_data[0][2,:])

In [ ]:
from smart import mesh_tools
import dolfin as d
import h5py

pitchArray=[0.0, 2.0, 3.0, 4.0, 5.0, 6.0]
radiusArray=[0.0, 0.2, 0.2, 0.2, 0.2, 0.2]
forceArray=[0.0, 400.0, 400.0, 800.0, 800.0]
t0Array=[100.0, 100.0, 1000.0, 100.0, 1000.0]
diffArray=[0.0, 0.001, 0.01, 0.1]
# actinArray=[0.0, 20.0, 0.0, 20.0]

var_names_saved = ["aNPC", "FActin", "J_NE", "J_Nuc", "LaminA", "LaminAp", "NE_alt_udef", "NE_udef", 
                   "NPC", "NPC_A", "Nuc_udef", "rupture_marker", "YAPTAZ_nuc"]
# var_names_saved = var_names_saved[1:]
plot_names = ["YAPTAZ_nuc", "LaminA", "LaminAp", "rupture_marker", "NPC_A"]
results_idx = []
for name in plot_names:
    results_idx.append(var_names_saved.index(name) + 1) # add one because time is first el

YAPTAZ_ratios = np.zeros([len(pitchArray),len(forceArray),len(diffArray)])
vol_vals = np.zeros([len(pitchArray),len(forceArray),len(diffArray)])
integrated_force = np.zeros([len(pitchArray),len(forceArray),len(diffArray)])
for curStr in ["pt5phiE"]:#,"pt5Lamin"]:
    if "Lamin" in curStr:
        parent_folder = f"/root/scratch/new_coupled_sims/results_nanopillars_{curStr}_2026-01-30_newPhos_phiE"
    else:
        parent_folder = f"/root/scratch/new_coupled_sims/results_nanopillars_{curStr}_2026-01-30_newPhos"
    for i in [4]:#range(len(pitchArray)):
        for j in [0,1,2]:#range(len(forceArray)):
            for k in [1]:#range(len(diffArray)):
                cur_folder = (f"{parent_folder}/nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                            f"force{forceArray[j]}_t0{t0Array[j]}_diff{diffArray[k]}")#_actin{actinArray[k]}")
                loaded_cur = mesh_tools.load_mesh(pathlib.Path(f"{cur_folder}/mesh/nuc_mesh.h5"))
                YAPTAZ_cur = np.loadtxt(pathlib.Path(f"{cur_folder}/YAPTAZ.txt"))
                vol_inner = np.loadtxt(pathlib.Path(f"{cur_folder}/mech_data/vol_inner.txt"))
                file_cur = (f"{cur_dir}/analysis_data/npy-files-{curStr}-submission/"#nuc-mechanics-coupled-corrected/"
                            f"nanopillars_r{radiusArray[i]}_p{pitchArray[i]}_"
                            f"force{forceArray[j]}_t0{t0Array[j]}_diff{diffArray[k]}_results_center_low.npy")#_actin{actinArray[k]}_results_all.npy")
                results_cur = np.load(file_cur)
                YAPnuc = results_cur[results_idx[0]]
                if radiusArray[i] == 0.0:
                    FActin0 = 140
                elif radiusArray[i] <= 0.15:
                    FActin0 = 80.0
                elif radiusArray[i] <= 0.25:
                    FActin0 = 100.0
                elif radiusArray[i] <= 0.5:
                    FActin0 = 120.0
                else:
                    FActin0 = 140.0
                Myo_A = 4.0
                K_CN = 4.0
                K_CY = 5.429e-3
                phiNP = (K_CN + K_CY*FActin0*Myo_A)/(1+ K_CN + K_CY*FActin0*Myo_A)
                YAPcyto = YAPTAZ_cur[:,1]/phiNP
                maxIdx = min([len(YAPnuc),len(YAPcyto)])
                YAPratio = YAPnuc[0:maxIdx] / YAPcyto[0:maxIdx]
                LaminA = results_cur[results_idx[1]]
                YAPTAZ_ratios[i,j,k] = YAPratio[-1]
                vol_vals[i,j,k] = vol_inner[-1]
                rupture_marker_vals = []
                lamin_vals = []
                rupture_file = h5py.File(f"{cur_folder}/rupture_marker.h5", 'r')
                lamin_file = h5py.File(f"{cur_folder}/LaminA.h5", 'r')
                for idx in range(len(vol_inner)-1):
                    try:
                        rupture_array = rupture_file['VisualisationVector'][f'{idx}'][()]
                        lamin_array = lamin_file['VisualisationVector'][f'{idx}'][()]
                    except:
                        break
                    rupture_marker_vals.append(np.max(rupture_array))
                    lamin_vals.append(np.min(lamin_array))
                # plt.plot(results_cur[0], YAPratio,
                #         label=f"{i},{j},{k}")
                # integrated_force[i,j,k] = np.trapz(results_cur[results_idx[3]],results_cur[0])
                integrated_force[i,j,k] = np.trapz(rupture_marker_vals, results_cur[0])
                # print(integrated_force[i][j][k])
                print(f"Max force per lamin is {max(rupture_marker_vals)} at time {results_cur[0][np.argmax(rupture_marker_vals)]}")
                plt.semilogx(results_cur[0], rupture_marker_vals,label=f"{i},{j},{k}")
                # plt.semilogx(results_cur[0], lamin_vals,label=f"{i},{j},{k}")
                # plt.semilogx(results_cur[0], results_cur[results_idx[3]],label=f"{i},{j},{k}")
        # plt.plot(vol_vals[i,:]/max(vol_vals[i,:]), YAPTAZ_ratios[i,:], 'o', label=substrate_names[i])

plt.legend()
plt.ylabel("Rupture vals")
plt.xlabel('Time (s)')
plt.xlim([1, 10000])
# plt.savefig("forcePerLamin_p5.pdf", format="pdf")

In [ ]:
np_means = {
    'Fast': integrated_force[:,1,1],#-integrated_force[0,1,1],
    'Slow': integrated_force[:,2,1],#-integrated_force[0,2,1],
}
width = 0.4
x = np.arange(len(pitchArray))
multiplier = 0
fig, ax = plt.subplots(layout='constrained', figsize=(5,4))
for attribute, measurement in np_means.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurement, width, label=attribute)#, hatch='///')
    # ax.bar_label(rects, padding=3)
    multiplier += 1
ax.set_ylabel('Integrated force per lamin')
ax.set_xticks(x + width/2, pitchArray)#[::2])
ax.set_xlabel('Pitch (µm)')
ax.legend()#loc='upper right')
plt.ylim([0,40000])
plt.savefig("forcePerLamin_lowLamin.pdf", format="pdf")

In [ ]:
var_names_saved = ["Cofilin_NP", "FActin", "GActin", "integrin", "integrin_L", "integrin_LT", "integrin_LTst", "integrin_tot",
                   "LaminA", "ligand", "LIMK_A", "mDia_A", "Myo_A", "NPC_A", "pFAK", "RhoA_GDP", "RhoA_GTP_Contact", "RhoA_GTP_PM", 
                   "ROCK_A", "YAPTAZ", "YAPTAZ_nuc", "YAPTAZ_phos"]
plot_names = ["YAPTAZ_phos", "YAPTAZ", "YAPTAZ_nuc"]
source_folder = "/root/shared/gitrepos/smart-mechanotransduction/analysis_data/npy-files-nuc-mechanics-coupled"
folder_list = os.listdir(source_folder)

for i in range(1,len(folder_list)):#[7, 10, 11, 12, 13, 14]:
    results_cur = np.load(f"{source_folder}/{folder_list[i]}")
    YAPTAZ_nuc = results_cur[10]
    # YAPratio_SS[i][j] = YAPratio[-1]
    # if i == 10:# and np.isclose(wasp_val, 0.01):
    if "p3.0" in folder_list[i]:
        plt.plot(results_cur[0], YAPTAZ_nuc, label=folder_list[i], linestyle="dashed")
    else:
        plt.plot(results_cur[0], YAPTAZ_nuc, label=folder_list[i])

# plt.legend()
plt.ylabel("Nuclear YAP/TAZ")
plt.xlabel('Time (s)')
plt.xlim([0, 1800])
# plt.savefig("cytosolic activation.pdf", format="pdf")

Plot super Gaussian functions used to approximate locations of nanopillars

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from matplotlib import cm
from matplotlib.ticker import LinearLocator

fig, ax = plt.subplots()#subplot_kw={"projection": "3d"})
# ax.view_init(elev=90, azim=0)

# Make data.
X = np.arange(0, 5, 0.01)
Y = np.arange(0, 5, 0.01)
X, Y = np.meshgrid(X, Y)

nucRad1 = 4.1
npSpacing = 2.0
npRad = 0.2 + 0.05
xMax = np.ceil(2*nucRad1 / npSpacing) * npSpacing
xNP = np.arange(0.0, xMax+1e-12, npSpacing)
yNP = np.arange(0.0, xMax+1e-12, npSpacing)
xNP, yNP = np.meshgrid(xNP, yNP)
xNP = xNP.flatten()
yNP = yNP.flatten()
nanopillar_logic = np.zeros_like(X)
for i in range(len(xNP)):
    # nanopillar_logic += exp(-((x[0]+u[0]-xNP[i])**2+(x[1]+u[1]-yNP[i])**2)/(2*npRad**2))
    nanopillar_logic += np.exp(-((X-xNP[i])**2+(Y-yNP[i])**2)**2/(npRad**4))

# Plot the surface.
domains = ax.imshow(nanopillar_logic)
plt.gca().invert_yaxis()
fig.colorbar(domains)#surf, shrink=0.5, aspect=5)

plt.savefig("200nm_domains.pdf", format="pdf")

In [ ]:
# plot cap stress over time
matplotlib.rcParams.update({'figure.figsize': (6,3)})
tVec = np.linspace(0.0, 1000.0)
plt.plot(tVec, (1-np.exp(-tVec/100.0)))
plt.savefig("cap_stress.pdf", format="pdf")